# 05. Sensitivity setup and methodology

This is the first of four sensitivity-analysis notebooks (05–08) for Paper 3. It bridges from the framework-comparison work in notebooks 01–04 to the computational sensitivity analysis: it sets out why sensitivity analysis is required, what analytical question the simulation answers, and what modelling stance the simulation takes. Subsequent notebooks present the primary results (06), robustness checks (07), and the discussion / implications (08).

**Epistemic Notice — Sensitivity Analysis Notebooks**

Unlike Paper 3's earlier notebooks (01–04), which render structured data extracted from the manuscript without generating new scientific claims, the notebooks in this sensitivity series (05–08) DO produce new computational findings under stated modelling assumptions. Specifically, they explore how variation in inter-rater agreement (κ) on the Negative Harm Test (NHT) propagates to four governance outcomes, calibrating κ tolerance regimes for the tier-classification rule.

This work is consistent with the Threshold Justification Stack's non-compensatory architecture. Each governance gate has its own threshold expressed in its own evidential "currency"; the framework rejects exchange rates between currencies. The κ-sensitivity simulation operates entirely within the NHT's own currency — it varies what the tier-classification reliability threshold should be, given the rule's purpose. It does not propose that κ performance on the NHT could compensate for thresholds in other gates of the framework.

Findings throughout this series are conditional on the modelling assumptions in [`docs/sensitivity/modelling_assumptions.md`](../docs/sensitivity/modelling_assumptions.md). Readers should treat results as methodological calibration evidence — informing what tier-classification reliability would need to look like for the NHT to function as designed — not as standalone empirical claims independent of the framework.

In [1]:
# Bootstrap: walk upward to find repo root, add to sys.path, then import tjs_sensitivity.*
# (mirrors P4's notebook pattern; the inline walk handles the chicken-and-egg of needing
# tjs_sensitivity importable before tjs_sensitivity.bootstrap can be imported)
import sys
from pathlib import Path

for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "config" / "harness_settings.json").is_file():
        _s = str(_cand)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break

from tjs_sensitivity.bootstrap import prepare_notebook, load_seed
from tjs_sensitivity.bootstrap import (
    SENSITIVITY_INPUTS_DIR,
    SENSITIVITY_OUTPUTS_DIR,
    SENSITIVITY_TABLES_DIR,
)

REPO_ROOT = prepare_notebook()
print(f"Repo root: {REPO_ROOT}")
print(f"Sensitivity inputs:  {REPO_ROOT / SENSITIVITY_INPUTS_DIR}")
print(f"Sensitivity outputs: {REPO_ROOT / SENSITIVITY_OUTPUTS_DIR}")

Repo root: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-3-framework-comparison
Sensitivity inputs:  C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-3-framework-comparison\inputs\sensitivity
Sensitivity outputs: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-3-framework-comparison\outputs\sensitivity


## 05.1 Why sensitivity analysis: bridging from framework

The framework-comparison work in notebooks 01–04 establishes the Threshold Justification Stack (TJS): six documentation layers for Primary Safety thresholds, three for Secondary Operational. The classification rule is the Negative Harm Test (NHT): a threshold may be classified Secondary Operational only if its failure cannot lead to clinical intervention or omission.

The sensitivity analysis is required because **the framework's tier-stratified documentation depth depends on tier-classification reliability for the proportionality argument to hold** (P3-C36). **If tier classification is unreliable, neither the burden reduction at the Secondary tier nor the proportionality of the documentation requirement is defensible** (P3-C53). The entire architecture rests on the premise that the NHT can be applied consistently.

This dependency is structural: **the TJS depends on consistent classification of thresholds as Primary Safety or Secondary Operational** (P3-C48). The sensitivity simulation in this series tests how robust the framework is to imperfect classification — not by measuring real-world κ (that would be empirical validation, deferred to the proposed pilot in §"Proposed pilot design with prespecified feasibility endpoints"), but by analysing what governance outcomes follow under stated modelling assumptions across a range of hypothetical κ values.

## 05.2 The Negative Harm Test — operational definition

The classification rule, expressed operationally, is: **a threshold may be classified Secondary Operational only if its failure cannot lead to clinical intervention or omission** (P3-C54). This is the boolean test that any single rater must make for any candidate threshold.

Two simulated raters apply this test independently per threshold. The output alphabet for each rater is `{Primary, Secondary, Uncertain}` — the third option captures cases where the rater cannot confidently apply the test. To avoid systematic risk-shifting, the adjudication rule is **default-to-Primary** on disagreement or Uncertain: if either rater says Primary, OR if either rater says Uncertain, the threshold is classified Primary.

This means the only path to a Secondary classification is `(rater_a == SECONDARY) AND (rater_b == SECONDARY)`. In all other cases the threshold receives the heavier Primary documentation requirement. The rule is conservative by design.

In [2]:
# Inspect the rater output alphabet (the three classifications)
from tjs_sensitivity.rater_model import PRIMARY, SECONDARY, UNCERTAIN

output_alphabet = (PRIMARY, SECONDARY, UNCERTAIN)
print(f"Rater output alphabet (size {len(output_alphabet)}): {output_alphabet}")

Rater output alphabet (size 3): ('Primary', 'Secondary', 'Uncertain')


## 05.3 Analytical question and modelling stance

The simulation answers a single analytical question: *under stated modelling assumptions, given an assumed inter-rater agreement κ on the NHT and the default-to-Primary adjudication rule, what governance outcomes follow* (P3-C47).

The four governance outcomes the simulation tracks:

1. **Misclassification rate (Primary → Secondary)**: the proportion of latent-Primary thresholds that the adjudicated rule classifies as Secondary. This is the safety-critical error: under-documentation of safety-critical thresholds.
2. **Misclassification rate (Secondary → Primary)**: the proportion of latent-Secondary thresholds that the adjudicated rule classifies as Primary. This is the over-documentation error: extra documentation burden for thresholds that don't need it.
3. **Net Primary rate**: the post-adjudication share of thresholds receiving the heavier Primary classification. Higher rates mean more documentation burden.
4. **Over-escalation burden** (hours per 100 thresholds): the documentation-burden delta caused by mis-classifications relative to perfect classification.

A critical epistemic posture: **the simulation is sensitivity analysis, not empirical estimation** (P3-C38). The κ values are scenario inputs, not measurements. The rater-error model is a mathematical abstraction calibrated to achieve target κ; it is not a model of how real raters actually err. The question is "what would follow if κ were 0.40 or 0.60 or 0.80", not "what is κ in any particular real institution".

## 05.4 The rater model: two simulated raters

The simulation places **two independent simulated raters per threshold** (P3-C37). Each rater independently:

1. With probability 0.02, returns `Uncertain`.
2. Otherwise, with probability `(1 - error_rate)` returns the latent true tier; with probability `error_rate` returns the opposite tier.

The Uncertain rate is held constant at 2% across all κ scenarios so that κ variation is driven purely by classification-error variation. The per-rater `error_rate` is calibrated by numerical inversion to achieve the target Cohen's κ at the given base rate.

In [3]:
# Load the κ grid, base-rate grid, and master seed
import json

kappa_grid_path = REPO_ROOT / SENSITIVITY_INPUTS_DIR / "kappa_grid.json"
base_rate_grid_path = REPO_ROOT / SENSITIVITY_INPUTS_DIR / "base_rate_grid.json"

with open(kappa_grid_path) as f:
    kappa_cfg = json.load(f)
with open(base_rate_grid_path) as f:
    base_rate_cfg = json.load(f)

master_seed = load_seed(REPO_ROOT)

print(f"Master seed (deterministic master): {master_seed}")
print(f"κ grid ({len(kappa_cfg['kappa_values'])} values): {kappa_cfg['kappa_values']}")
print(f"Base-rate grid ({len(base_rate_cfg['base_rate_primary_values'])} values): "
      f"{base_rate_cfg['base_rate_primary_values']}")
print(f"Total scenarios: {len(kappa_cfg['kappa_values']) * len(base_rate_cfg['base_rate_primary_values'])}")

Master seed (deterministic master): 20260426
κ grid (7 values): [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
Base-rate grid (4 values): [0.05, 0.1, 0.2, 0.5]
Total scenarios: 28


In [4]:
# Demonstrate the rater model: simulate two raters at κ=0.60, π_Primary=0.10, n=1000
import numpy as np
from tjs_sensitivity.rater_model import simulate_raters, cohen_kappa_binary

# Generate a synthetic latent population (1000 thresholds, 10% Primary)
rng = np.random.default_rng(master_seed)
n_thresholds = 1000
base_rate = 0.10
target_kappa = 0.60

is_primary = rng.random(n_thresholds) < base_rate
latent = np.where(is_primary, PRIMARY, SECONDARY).astype(object)

# Simulate two raters at the target κ
rater_a, rater_b = simulate_raters(
    latent_true=latent,
    target_kappa=target_kappa,
    base_rate_primary=base_rate,
    uncertain_prob=0.02,
    rng=rng,
)

# Verify realised κ matches target
realised_kappa = cohen_kappa_binary(rater_a, rater_b)
print(f"Target κ: {target_kappa:.2f}")
print(f"Realised κ (single replicate, n=1000): {realised_kappa:.3f}")
print(f"Distribution of rater_a: Primary={np.sum(rater_a == PRIMARY)}, "
      f"Secondary={np.sum(rater_a == SECONDARY)}, "
      f"Uncertain={np.sum(rater_a == UNCERTAIN)}")

Target κ: 0.60
Realised κ (single replicate, n=1000): 0.000
Distribution of rater_a: Primary=120, Secondary=863, Uncertain=17


In [5]:
# Show the analytical mapping from target κ to per-rater error rate
from tjs_sensitivity.rater_model import kappa_to_error_rate

print("Target κ → per-rater error rate (at π_Primary = 0.10, uncertain_prob = 0.02):")
print(f"{'target κ':>10}  {'error rate':>12}")
for k in kappa_cfg['kappa_values']:
    err = kappa_to_error_rate(k, base_rate_primary=0.10, uncertain_prob=0.02)
    print(f"{k:>10.2f}  {err:>12.4f}")

Target κ → per-rater error rate (at π_Primary = 0.10, uncertain_prob = 0.02):
  target κ    error rate
      0.20        0.1701
      0.30        0.1195
      0.40        0.0836
      0.50        0.0565
      0.60        0.0353
      0.70        0.0181
      0.80        0.0039


## 05.5 Pass criteria

This setup notebook verifies the following invariants. Failure of any assertion below is a setup-level reproducibility failure — it means the simulation inputs or the rater model are not what the manuscript and `docs/sensitivity/simulation_design.md` describe.

The asserts cover:

- The master seed parses to the expected value (`20260426`).
- The κ grid matches the manuscript-stated values `{0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80}`.
- The base-rate grid matches `{0.05, 0.10, 0.20, 0.50}`.
- The rater output alphabet has exactly three elements `{Primary, Secondary, Uncertain}`.
- The κ-to-error-rate analytical inversion produces a monotonically decreasing error rate as target κ increases (higher reliability ⇒ lower error rate).

In [6]:
# Pass criterion asserts (this notebook fails loudly if any setup invariant is broken)

# 1. Master seed
assert master_seed == 20260426, f"Master seed mismatch: got {master_seed}, expected 20260426"

# 2. κ grid
expected_kappas = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
loaded_kappas = kappa_cfg['kappa_values']
assert loaded_kappas == expected_kappas, (
    f"κ grid mismatch: got {loaded_kappas}, expected {expected_kappas}"
)

# 3. Base-rate grid
expected_base_rates = [0.05, 0.10, 0.20, 0.50]
loaded_base_rates = base_rate_cfg['base_rate_primary_values']
assert loaded_base_rates == expected_base_rates, (
    f"Base-rate grid mismatch: got {loaded_base_rates}, expected {expected_base_rates}"
)

# 4. Rater output alphabet
assert len(output_alphabet) == 3, f"Rater alphabet size mismatch: {len(output_alphabet)}"
assert set(output_alphabet) == {"Primary", "Secondary", "Uncertain"}, (
    f"Rater alphabet content mismatch: {output_alphabet}"
)

# 5. Monotonic κ → error rate (higher κ = lower error)
error_rates = [
    kappa_to_error_rate(k, base_rate_primary=0.10, uncertain_prob=0.02)
    for k in expected_kappas
]
for i in range(len(error_rates) - 1):
    assert error_rates[i] >= error_rates[i + 1], (
        f"Non-monotonic κ→error mapping at index {i}: "
        f"κ={expected_kappas[i]:.2f}→{error_rates[i]:.4f}, "
        f"κ={expected_kappas[i+1]:.2f}→{error_rates[i+1]:.4f}"
    )

print("ALL PASS-CRITERION ASSERTS HOLD")
print("Setup invariants verified:")
print(f"  - master seed: {master_seed}")
print(f"  - κ grid: {expected_kappas}")
print(f"  - base-rate grid: {expected_base_rates}")
print(f"  - rater alphabet: {set(output_alphabet)}")
print(f"  - κ→error rate monotonic: ✓")

ALL PASS-CRITERION ASSERTS HOLD
Setup invariants verified:
  - master seed: 20260426
  - κ grid: [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
  - base-rate grid: [0.05, 0.1, 0.2, 0.5]
  - rater alphabet: {'Secondary', 'Primary', 'Uncertain'}
  - κ→error rate monotonic: ✓


---

**Notebook 05 complete.** Setup verified; the framework dependency on tier-classification reliability is established (P3-C53, P3-C48); the operational NHT classification rule is documented (P3-C54); the analytical question is framed (P3-C47); the modelling stance (P3-C38, P3-C36) is declared; the rater model (P3-C37) is loaded and demonstrated.

The next notebook (`06_sensitivity_primary_results.ipynb`) runs the full Monte Carlo sweep across the 28 (κ, π) scenarios and derives the three κ regimes (falsification κ < 0.40; caution 0.40 ≤ κ < 0.60; operational κ ≥ 0.60) at the manuscript's primary base rate π_Primary = 0.10.